# Test for Multi-layers

This paper illustrates the methodology (initially developped within the package Isatoil) which enables a joint estimation of a set of surfaces in a layer-cake framework. This methodology is based on a multi-variate approach (as many variables as they are surfaces). It is based on the information provided by the intercepts of well data with those surfaces. The spatial characteristics of the surfaces are captured in a multivariate model defined in covariances. This technique enables dealing with any kind of well information (vertical,
inclined, deviated or horizontal) as it relies on the joint estimation of cumulative layer thicknesses (True Vertical Depth).

The method offers some interesting properties, such as:
- to define the reference surface which serves as a support to the whole layer cake system
- to establish the spatial multivariate model either on thicknesses or on velocities (if exhaustive time maps are provided). The results are then produced either in cumulative thickness maps or in layer velocity maps
- to constraint the whole layer-cake package not to exceed a bottom map (specified exhaustively)

In [ ]:
import gstlearn as gl
import pandas as pd
import gstlearn.plot as gp
import gstlearn.document as gdoc
import matplotlib.pyplot as plt

gdoc.setNoScroll()

We define the global parameters used in all the displays.

In [ ]:
cex = 1
pch = 21
ol = "white"
nbtuba = 1000

## Basic Methodology

This case study performs the different types of multilayer estimations with different hypotheses. All the estimations are represented through a vertical section across the field.

### Defining the Environment

A second part of the environment consists in the set of 4 colors used to represent the layers. This color scale will be stored in the local variable colors.

In [ ]:
colors = ["yellow", "orange", "red", "purple"]

### Creating Data Bases

The first task is to define the well information: this consists in the intercepts with the different surfaces. Data are characterized by their 2-D coordinates, the elevation of the intercept and the rank of the intercepted variable (locator: layer). The total number of surfaces is equal to 4. The following paragraph describes the contents of the ASCII data file.

In [ ]:
datcsv = pd.read_csv(gdoc.loadData("MLayers", "Intercepts.csv"))
db = gl.Db_fromPandas(datcsv)
db.setLocators(["x1", "x2"], gl.ELoc.X)
db.setLocators(["z1"], gl.ELoc.Z)
db.setLocators(["layer"], gl.ELoc.LAYER)
db

All the case study will be performed using a Unique Neighborhood which is created here and saved in the object called **neigh**.

In [ ]:
neigh = gl.NeighUnique()

A 2-D grid is created covering the area of interest. 

In [ ]:
grid = gl.DbGrid.create(nx=[100, 100])

Several variables are created in the Grid file which will be used to demonstrate some of the options:
- a surface is generated as a tilted plane covering the field: it will possibly be used as the reference surface
(hence its name)

In [ ]:
err = grid.addColumns((grid["x1"] + grid["x2"]) / 200, "reference")

- a more irregular surface is generated which will possibly be used as the bottom surface (hence its name). This surface is simulated using a simulation (with a short range spherical model), conditioned by the well intercepts with the layer #4. Ordinary Kriging is used to set the mean to the average of the data.

In [ ]:
model = gl.Model.createFromParam(gl.ECov.SPHERICAL, range=10, sill=0.05)
model.setDriftIRF(0)
db.addSelection(db["layer"] == 4)
err = gl.simtub(db, grid, model, neigh, seed=0, nbtuba=nbtuba)
grid.setName("Simu.z1", "bottom")
db.clearSelection()

- a set of 4 time surfaces are generated: they are simulated unconditionally in the time system, with
different means (1000m/s for the first one, 1200 for the second one, 1400 for the third one and 1600
for the last one). For simplicity, they all follow the same spatial characteristics: short range spherical
model with the same variance

In [ ]:
model = gl.Model.createFromParam(gl.ECov.SPHERICAL, range=10, sill=4)

model.setMean(1000)
err = gl.simtub(None, grid, model, seed=0, nbtuba=nbtuba)
grid.setName("Simu", "Time1000")

model.setMean(1200)
err = gl.simtub(None, grid, model, seed=0, nbtuba=nbtuba)
grid.setName("Simu", "Time1200")

model.setMean(1400)
err = gl.simtub(None, grid, model, seed=0, nbtuba=nbtuba)
grid.setName("Simu", "Time1400")

model.setMean(1600)
err = gl.simtub(None, grid, model, seed=0, nbtuba=nbtuba)
grid.setName("Simu", "Time1600")

grid.setLocators(["Time*"], gl.ELoc.TIME)

In [ ]:
grid

A trace (stored in the object trace) is defined which crosses the field along its first diagonal. 

In [ ]:
trace = gl.MatrixDense.createFromVD([0, 50, 90, 0, 50, 90], 3, 2)
trace

In the next figure, we represent the data information on a horizontal view plane, together with the trace.

In [ ]:
fix, ax = plt.subplots()
ax.symbol(db, nameColor="layer", c=colors, s=100)
ax.XY(trace.getColumn(0), trace.getColumn(1))
plt.show()

The multivariate model used in all subsequent interpolations is defined here. For simplicity sake, all the variables are independent and follow the same spatial characteristics (with a cubic variogram) with different sills

In [ ]:
sills = gl.MatrixSymmetric.createFromDiagonal([1.0, 3.0, 2.0, 4.0])
model = gl.Model.createFromParam(gl.ECov.CUBIC, range=40, sills=sills)

## Multilayers estimation

In [ ]:
figsize = [10, 4]
checkOrder = 1
edgecolor = "black"

It is now time to perform the joint estimation of the set of surfaces of the layer cake. For all the different options, we store the results in an auxiliary Grid file (called a) and display the them along the vertical section (with the vertical axis oriented downwards).

The first attempt is the standard interpolation when defining no reference not bottom surfaces.

In [ ]:
err = gl.multilayers_kriging(db, grid, model, namconv=gl.NamingConvention("MLKriging"))

The next figure shows the **natural** estimation of the different surfaces. Note that there is no requirement to forbid their intersection (far from the control points).

In [ ]:
gp.init(figsize=figsize)
gp.sectionFromGrid(trace, grid, flagFill=False, flagUp=False, colors=colors)
gp.sectionFromPoints(trace, db, colors=colors, s=100, edgecolor=edgecolor)
gp.decoration("Depth Estimation")

If we fill the inter-surface areas, the visibility is improved.

In [ ]:
gp.init(figsize=figsize)
gp.sectionFromGrid(
    trace, grid, flagFill=True, flagUp=False, colors=colors, checkOrder=checkOrder
)
gp.sectionFromPoints(trace, db, colors=colors, s=100, edgecolor=edgecolor)
gp.decoration("Depth Estimation (painted by layers)")

In the following case, a reference surface is added and the same interpolation is carried on (all thicknesses are calculated in the flattened system).

In [ ]:
err = gl.multilayers_kriging(
    db, grid, model, namerefd="reference", namconv=gl.NamingConvention("MLKrigRef")
)

In [ ]:
gp.init(figsize=figsize)
gp.sectionFromGrid(
    trace,
    grid,
    addNames={"reference"},
    flagFill=True,
    flagUp=False,
    colors=colors,
    checkOrder=checkOrder,
)
gp.sectionFromPoints(trace, db, colors=colors, s=100, edgecolor="black")
gp.decoration("Depth Estimation (with tilted Reference surface)")

The results of the previous attempt are presented in layer thickness. Obviously it does not make sense to overlay the well intercepts provided in depth.

In [ ]:
err = gl.multilayers_kriging(
    db,
    grid,
    model,
    namerefd="reference",
    flag_Z=False,
    namconv=gl.NamingConvention("MLKrigThick"),
)

In [ ]:
gp.init(figsize=figsize)
gp.sectionFromGrid(trace, grid, flagFill=False, flagUp=True, colors=colors)
gp.decoration("Thickness Estimation")

The same interpolation can be performed considering that the spatial characteristics correspond to the velocities (rather than to thicknesses)

In [ ]:
err = gl.multilayers_kriging(
    db,
    grid,
    model,
    namerefd="reference",
    flag_vel=True,
    namconv=gl.NamingConvention("MLKrigVel"),
)

In [ ]:
gp.init(figsize=figsize)
gp.sectionFromGrid(
    trace,
    grid,
    addNames={"reference"},
    flagFill=True,
    flagUp=False,
    colors=colors,
    checkOrder=checkOrder,
)
gp.sectionFromPoints(trace, db, colors=colors, s=100, edgecolor=edgecolor)
gp.decoration("Depth Estimation (performed using Velocities)")

This last trial produces the layer velocities obtained in the same circonstances:

In [ ]:
err = gl.multilayers_kriging(
    db,
    grid,
    model,
    namerefd="reference",
    flag_vel=True,
    flag_Z=False,
    namconv=gl.NamingConvention("MLKrigVel"),
)

In [ ]:
gp.init(figsize=figsize)
gp.sectionFromGrid(trace, grid, flagFill=False, flagUp=True, colors=colors)
gp.decoration("Velocity Estimation")

In this last trial, we define both the reference and the bottom surfaces and perform the interpolation in depth. These two external surfaces are also displayed.

In [ ]:
err = gl.multilayers_kriging(
    db,
    grid,
    model,
    namerefd="reference",
    namerefb="bottom",
    namconv=gl.NamingConvention("MLKrigRefBot"),
)

In [ ]:
gp.init(figsize=figsize)
gp.sectionFromGrid(
    trace,
    grid,
    addNames={"reference", "bottom"},
    flagFill=True,
    flagUp=False,
    colors=colors,
    checkOrder=checkOrder,
)
gp.sectionFromPoints(trace, db, colors=colors, s=100, edgecolor=edgecolor)
gp.decoration("Depth Estimation (using Bottom Constraints and Reference)")

## Bayesian Kriging

The procedure is continued by adding the relevant output when using some Bayes criterion. Compared to the usual case, some additional information is printed out using the keypair mechanism.

In [ ]:
err = gl.multilayers_getPrior(db, grid, model)

#    err = gl.multilayers_kriging(db,grid,model,namerefd="reference", namerefb = "bottom",
#                           namconv=gl.NamingConvention("MLKrigRefBot"))

# Debugging principle

In this chapter, we print the relevant information to allow debugging. Some additional ingredients have been left apart, such as the use of some reference surfaces both in depth and in time.

### Standard Kriging

In this first part, we concentrate on the kriging system. The whole system is dumped out when processing
the first grid node (named "1"). It is displayed when working:
- in depth

In [ ]:
gl.OptDbg.setReference(1)

In [ ]:
err = gl.multilayers_kriging(
    db, grid, model, flag_vel=False, namconv=gl.NamingConvention("MLTest")
)

- or in velocities

In [ ]:
err = gl.multilayers_kriging(
    db, grid, model, flag_vel=True, namconv=gl.NamingConvention("MLTest")
)

In [ ]:
gl.OptDbg.setReference(-1)